In [1]:

from pathlib import Path
import re
import shutil
import subprocess
import difflib

import pandas as pd
import ipywidgets as widgets
from IPython.display import display, Markdown, HTML, clear_output

# ============================================================
# Configuration
# ============================================================

NAMESPACE = "web01"

PRODUCTION_ENTRYPOINT = Path("/etc/nftables.conf")
TEST_ENTRYPOINT = Path("/tmp/nftables-zones-test.conf")

VARIABLES_INCLUDE = "/etc/nft/web01/variables/*.nft"
CONSTANTS_INCLUDE = "/etc/nft/web01/constants/*.nft"
ZONES_INCLUDE = "/etc/nft/web01/zones/*.nft"

ZONE_IDS = ("01", "02", "03")

PROTOCOLS = {
    "TCP": {
        "source": Path("/etc/nft/web01/rules/10-tcp-networks.nft"),
        "staging": Path("/tmp/10-tcp-network.nft"),
        "field": "port",
        "label": "Port/service",
    },
    "UDP": {
        "source": Path("/etc/nft/web01/rules/08-udp-networks.nft"),
        "staging": Path("/tmp/08-udp-networks.nft"),
        "field": "port",
        "label": "Port/service",
    },
    "ICMP": {
        "source": Path("/etc/nft/web01/rules/06-icmp-networks.nft"),
        "staging": Path("/tmp/06-icmp-network.nft"),
        "field": "icmp_type",
        "label": "ICMP type",
    },
}

DEFAULTS = {
    "interface": "$nic_lan",
    "source": "$ip_lan",
    "destination": "$net_lan01",
    "port": "",
    "icmp_type": "icmp",
    "verdict": "accept",
}

ALLOWED_VERDICTS = {"accept", "drop", "reject"}

original_sources = {}
rules_by_zone_protocol = {
    zone: {protocol: [] for protocol in PROTOCOLS}
    for zone in ZONE_IDS
}
staged_sources = {}


# ============================================================
# Read source files
# ============================================================

def read_source(protocol):
    path = PROTOCOLS[protocol]["source"]

    if not path.is_file():
        raise FileNotFoundError(
            f"Cannot read {protocol} source file: {path}"
        )

    return path.read_text()


# ============================================================
# Parse definitions, including empty definitions
# ============================================================

def parse_definition(source, name):
    pattern = (
        r"(?m)^[ \t]*define[ \t]+"
        + re.escape(name)
        + r"[ \t]*=[ \t]*\{([^}]*)\}"
    )

    matches = list(re.finditer(pattern, source, flags=re.DOTALL))

    if len(matches) != 1:
        raise ValueError(
            f"Expected exactly one definition of {name}; "
            f"found {len(matches)}."
        )

    entries = []

    for line_number, raw_line in enumerate(
        matches[0].group(1).splitlines(), start=1
    ):
        line = raw_line.strip()

        if not line or line.startswith("#"):
            continue

        entry_match = re.fullmatch(
            r"(.+?)\s*:\s*([A-Za-z_][A-Za-z0-9_]*)\s*,?",
            line,
        )

        if not entry_match:
            raise ValueError(
                f"Cannot parse entry in {name}, body line "
                f"{line_number}: {raw_line!r}"
            )

        entries.append({
            "fields": [
                field.strip()
                for field in entry_match.group(1).strip().split(".")
            ],
            "verdict": entry_match.group(2).strip().lower(),
        })

    return entries


# ============================================================
# Zone-specific key layouts
# ============================================================

def zone_key_layout(zone, protocol):
    if zone in ("01", "02"):
        return {
            "r01_count": 5,
            "r02_count": 4,
            "has_zone_prefix": True,
            "has_final_field": True,
        }

    if zone == "03":
        if protocol in ("TCP", "UDP"):
            return {
                "r01_count": 5,
                "r02_count": 4,
                "has_zone_prefix": True,
                "has_final_field": True,
            }

        return {
            "r01_count": 4,
            "r02_count": 3,
            "has_zone_prefix": True,
            "has_final_field": False,
        }

    raise ValueError(f"Unsupported zone: {zone}")


# ============================================================
# Load paired definitions for a zone and protocol
# ============================================================

def load_zone_protocol(source, protocol, zone):
    suffix = {
        "TCP": "TCP4",
        "UDP": "UDP4",
        "ICMP": "ICMP4",
    }[protocol]

    r01_name = f"Z{zone}_{suffix}_R01"
    r02_name = f"Z{zone}_{suffix}_R02"

    r01 = parse_definition(source, r01_name)
    r02 = parse_definition(source, r02_name)

    if len(r01) != len(r02):
        raise ValueError(
            f"Zone {zone} {protocol}: R01 has {len(r01)} entries "
            f"but R02 has {len(r02)}."
        )

    layout = zone_key_layout(zone, protocol)
    rules = []

    for number, (left, right) in enumerate(
        zip(r01, r02), start=1
    ):
        fields01 = left["fields"]
        fields02 = right["fields"]

        if len(fields01) != layout["r01_count"]:
            raise ValueError(
                f"Zone {zone} {protocol} rule {number}: "
                f"R01 expected {layout['r01_count']} fields, "
                f"found {fields01!r}."
            )

        if len(fields02) != layout["r02_count"]:
            raise ValueError(
                f"Zone {zone} {protocol} rule {number}: "
                f"R02 expected {layout['r02_count']} fields, "
                f"found {fields02!r}."
            )

        if fields01[0] != f"$Z{zone}":
            raise ValueError(
                f"Zone {zone} {protocol} rule {number}: "
                f"expected $Z{zone}, found {fields01[0]!r}."
            )

        if fields01[1:] != fields02:
            raise ValueError(
                f"Zone {zone} {protocol} rule {number}: "
                f"paired keys differ.\n"
                f"R01: {fields01[1:]!r}\n"
                f"R02: {fields02!r}"
            )

        if left["verdict"] != right["verdict"]:
            raise ValueError(
                f"Zone {zone} {protocol} rule {number}: "
                f"paired verdicts differ: "
                f"{left['verdict']!r} vs {right['verdict']!r}."
            )

        if layout["has_final_field"]:
            rule = {
                "interface": fields01[1],
                "source": fields01[2],
                "destination": fields01[3],
                "verdict": left["verdict"],
            }

            if protocol == "ICMP":
                rule["icmp_type"] = fields01[4]
            else:
                rule["port"] = fields01[4]

        else:
            # Zone 03 ICMP:
            # R01 = zone . interface . destination . icmp
            # R02 = interface . destination . icmp
            rule = {
                "interface": fields01[1],
                "source": "",
                "destination": fields01[2],
                "verdict": left["verdict"],
                "icmp_type": fields01[3],
            }

        rules.append(rule)

    return rules


# ============================================================
# Initialize all zones and protocols
# ============================================================

original_sources.clear()
staged_sources.clear()

rules_by_zone_protocol = {
    zone: {protocol: [] for protocol in PROTOCOLS}
    for zone in ZONE_IDS
}

for protocol in PROTOCOLS:
    source = read_source(protocol)
    original_sources[protocol] = source

    for zone in ZONE_IDS:
        rules_by_zone_protocol[zone][protocol] = (
            load_zone_protocol(source, protocol, zone)
        )

print("Zone editor configuration loaded successfully.")

for zone in ZONE_IDS:
    print(f"\nZone {zone}:")
    for protocol in PROTOCOLS:
        print(
            f"  {protocol}: "
            f"{len(rules_by_zone_protocol[zone][protocol])} rules"
        )

print(f"\nNamespace: {NAMESPACE}")
print(f"Test entry point: {TEST_ENTRYPOINT}")
print(f"Production entry point: {PRODUCTION_ENTRYPOINT}")


Zone editor configuration loaded successfully.

Zone 01:
  TCP: 10 rules
  UDP: 5 rules
  ICMP: 4 rules

Zone 02:
  TCP: 10 rules
  UDP: 15 rules
  ICMP: 5 rules

Zone 03:
  TCP: 4 rules
  UDP: 2 rules
  ICMP: 2 rules

Namespace: web01
Test entry point: /tmp/nftables-zones-test.conf
Production entry point: /etc/nftables.conf


In [2]:

# ============================================================
# Map rendering
# ============================================================

def key_r01(protocol, rule, zone):
    if zone == "03" and protocol == "ICMP":
        return " . ".join([
            "$Z03",
            rule["interface"],
            rule["destination"],
            rule["icmp_type"],
        ])

    final_field = (
        rule["icmp_type"] if protocol == "ICMP" else rule["port"]
    )

    return " . ".join([
        f"$Z{zone}",
        rule["interface"],
        rule["source"],
        rule["destination"],
        final_field,
    ])


def key_r02(protocol, rule, zone):
    if zone == "03" and protocol == "ICMP":
        return " . ".join([
            rule["interface"],
            rule["destination"],
            rule["icmp_type"],
        ])

    final_field = (
        rule["icmp_type"] if protocol == "ICMP" else rule["port"]
    )

    return " . ".join([
        rule["interface"],
        rule["source"],
        rule["destination"],
        final_field,
    ])


def nft_entry(key, verdict):
    return f"{key} : {verdict},"


def render_definition(name, rules, key_function, protocol, zone):
    if not rules:
        return f"define {name} = {{}}"

    lines = [f"define {name} = {{"]

    for rule in rules:
        lines.append(
            f"    {nft_entry(key_function(protocol, rule, zone), rule['verdict'])}"
        )

    lines.append("}")
    return "\n".join(lines)


# ============================================================
# Validation
# ============================================================

def validate_protocol(zone, protocol):
    errors = []
    rules = rules_by_zone_protocol[zone][protocol]
    seen = set()

    for number, rule in enumerate(rules, start=1):
        label = f"Zone {zone} {protocol} rule {number}"

        required = [
            "interface",
            "destination",
            "verdict",
            "icmp_type" if protocol == "ICMP" else "port",
        ]

        if not (zone == "03" and protocol == "ICMP"):
            required.append("source")

        for field in required:
            value = rule.get(field, "")

            if not isinstance(value, str) or not value.strip():
                errors.append(f"{label}: {field} is empty.")
                continue

            if any(
                character in value
                for character in ("\n", "\r", ",", ";", "{", "}")
            ):
                errors.append(
                    f"{label}: invalid delimiter in {field}."
                )

            if ":" in value:
                errors.append(
                    f"{label}: unexpected colon in {field}."
                )

        if protocol in ("TCP", "UDP"):
            port = rule["port"]

            valid_port = (
                (port.isdigit() and 1 <= int(port) <= 65535)
                or bool(
                    re.fullmatch(
                        r"\$[A-Za-z_][A-Za-z0-9_]*", port
                    )
                )
                or bool(
                    re.fullmatch(
                        r"[A-Za-z_][A-Za-z0-9_-]*", port
                    )
                )
            )

            if not valid_port:
                errors.append(
                    f"{label}: invalid port/service {port!r}."
                )

        else:
            icmp_type = rule["icmp_type"]

            if not re.fullmatch(
                r"[A-Za-z_][A-Za-z0-9_-]*", icmp_type
            ):
                errors.append(
                    f"{label}: invalid ICMP type {icmp_type!r}."
                )

        if rule["verdict"] not in ALLOWED_VERDICTS:
            errors.append(f"{label}: invalid verdict.")

        signature = tuple(
            rule.get(field, "")
            for field in required
        )

        if signature in seen:
            errors.append(f"{label}: duplicate rule.")

        seen.add(signature)

        if (
            key_r01(protocol, rule, zone).split(" . ")[1:]
            != key_r02(protocol, rule, zone).split(" . ")
        ):
            errors.append(f"{label}: R01/R02 key mismatch.")

    return errors


# ============================================================
# Editor widgets
# ============================================================

zone_selector = widgets.ToggleButtons(
    options=[
        ("Zone 01", "01"),
        ("Zone 02", "02"),
        ("Zone 03", "03"),
    ],
    value="01",
    description="Zone:",
)

protocol_selector = widgets.ToggleButtons(
    options=["TCP", "UDP", "ICMP"],
    value="TCP",
    description="Protocol:",
)

rule_selector = widgets.Dropdown(description="Rule:")

interface_input = widgets.Text(description="Interface:")
source_input = widgets.Text(description="Source:")
destination_input = widgets.Text(description="Destination:")

port_input = widgets.Text(description="Port/service:")
icmp_type_input = widgets.Text(
    description="ICMP type:",
    value="icmp",
)

verdict_input = widgets.Dropdown(
    options=["accept", "drop", "reject"],
    value="accept",
    description="Verdict:",
)

add_button = widgets.Button(
    description="Add", button_style="success"
)
update_button = widgets.Button(
    description="Update", button_style="primary"
)
delete_button = widgets.Button(
    description="Delete", button_style="danger"
)

table_output = widgets.Output()
status_output = widgets.Output()
workflow_output = widgets.Output()
diff_output = widgets.Output()

stage_button = widgets.Button(
    description="Check All Zones and Protocols",
    button_style="info",
)

deploy_button = widgets.Button(
    description="Deploy and Reload",
    button_style="warning",
    disabled=True,
)


def active_zone():
    return zone_selector.value


def active_protocol():
    return protocol_selector.value


def active_rules():
    return rules_by_zone_protocol[active_zone()][active_protocol()]


def set_status(message, error=False):
    with status_output:
        clear_output(wait=True)
        color = "#b91c1c" if error else "#166534"
        display(HTML(f"<p style='color:{color}'>{message}</p>"))


def refresh_editor(*_):
    zone = active_zone()
    protocol = active_protocol()
    rules = active_rules()

    rule_selector.options = [
        (
            f"Rule {i}: "
            f"{rule.get('icmp_type', rule.get('port', ''))}",
            i - 1,
        )
        for i, rule in enumerate(rules, start=1)
    ]

    icmp_type_input.layout.display = (
        "" if protocol == "ICMP" else "none"
    )
    port_input.layout.display = (
        "none" if protocol == "ICMP" else ""
    )

    source_input.layout.display = (
        "none" if zone == "03" and protocol == "ICMP" else ""
    )

    with table_output:
        clear_output(wait=True)
        rows = []

        for number, rule in enumerate(rules, start=1):
            final_field = (
                rule["icmp_type"]
                if protocol == "ICMP"
                else rule["port"]
            )

            rows.append({
                "#": number,
                "Interface": rule["interface"],
                "Source": rule.get("source", ""),
                "Destination": rule["destination"],
                PROTOCOLS[protocol]["label"]: final_field,
                "Verdict": rule["verdict"],
                "R01": nft_entry(
                    key_r01(protocol, rule, zone), rule["verdict"]
                ),
                "R02": nft_entry(
                    key_r02(protocol, rule, zone), rule["verdict"]
                ),
            })

        display(Markdown(f"### Zone {zone} — {protocol}"))
        display(pd.DataFrame(rows) if rows else "No rules defined.")

    load_selected_rule()


def load_selected_rule(*_):
    index = rule_selector.value
    protocol = active_protocol()
    zone = active_zone()
    rules = active_rules()

    if index is None or index >= len(rules):
        interface_input.value = DEFAULTS["interface"]
        source_input.value = ""
        destination_input.value = DEFAULTS["destination"]
        port_input.value = DEFAULTS["port"]
        icmp_type_input.value = DEFAULTS["icmp_type"]
        verdict_input.value = DEFAULTS["verdict"]

        if not (zone == "03" and protocol == "ICMP"):
            source_input.value = DEFAULTS["source"]

        return

    rule = rules[index]

    interface_input.value = rule["interface"]
    source_input.value = rule.get("source", "")
    destination_input.value = rule["destination"]
    verdict_input.value = rule["verdict"]

    if protocol == "ICMP":
        icmp_type_input.value = rule["icmp_type"]
    else:
        port_input.value = rule["port"]


def form_rule():
    protocol = active_protocol()
    zone = active_zone()

    rule = {
        "interface": interface_input.value.strip(),
        "destination": destination_input.value.strip(),
        "verdict": verdict_input.value,
    }

    if not (zone == "03" and protocol == "ICMP"):
        rule["source"] = source_input.value.strip()
    else:
        rule["source"] = ""

    if protocol == "ICMP":
        rule["icmp_type"] = icmp_type_input.value.strip()
    else:
        rule["port"] = port_input.value.strip()

    required = [
        "interface",
        "destination",
        "verdict",
        "icmp_type" if protocol == "ICMP" else "port",
    ]

    if not (zone == "03" and protocol == "ICMP"):
        required.append("source")

    for field in required:
        value = rule[field]

        if not value:
            raise ValueError(f"{field} cannot be empty.")

        if any(
            character in value
            for character in ("\n", "\r", ",", ";", "{", "}")
        ):
            raise ValueError(f"Invalid character in {field}.")

        if ":" in value:
            raise ValueError(f"Unexpected colon in {field}.")

    return rule


def add_rule(_):
    try:
        rule = form_rule()

        if rule in active_rules():
            raise ValueError("Duplicate rule.")

        active_rules().append(rule)
        refresh_editor()
        deploy_button.disabled = True
        set_status(
            f"Zone {active_zone()} {active_protocol()} rule added in memory."
        )

    except Exception as exc:
        set_status(str(exc), error=True)


def update_rule(_):
    try:
        index = rule_selector.value

        if index is None:
            raise ValueError("Select a rule.")

        rule = form_rule()
        rules = active_rules()

        if any(
            i != index and item == rule
            for i, item in enumerate(rules)
        ):
            raise ValueError("Duplicate rule.")

        rules[index] = rule
        refresh_editor()
        deploy_button.disabled = True
        set_status(
            f"Zone {active_zone()} {active_protocol()} rule updated in memory."
        )

    except Exception as exc:
        set_status(str(exc), error=True)


def delete_rule(_):
    index = rule_selector.value

    if index is None:
        set_status("Select a rule.", error=True)
        return

    active_rules().pop(index)
    refresh_editor()
    deploy_button.disabled = True
    set_status(
        f"Zone {active_zone()} {active_protocol()} rule deleted."
    )


zone_selector.observe(refresh_editor, names="value")
protocol_selector.observe(refresh_editor, names="value")
rule_selector.observe(load_selected_rule, names="value")

add_button.on_click(add_rule)
update_button.on_click(update_rule)
delete_button.on_click(delete_rule)

display(Markdown("# Zone Editor — Zones 01, 02 and 03"))
display(widgets.HBox([zone_selector, protocol_selector]))
display(table_output)

display(widgets.VBox([
    rule_selector,
    interface_input,
    source_input,
    destination_input,
    port_input,
    icmp_type_input,
    verdict_input,
    widgets.HBox([
        add_button,
        update_button,
        delete_button,
    ]),
    status_output,
]))

display(widgets.HBox([stage_button, deploy_button]))
display(workflow_output)
display(diff_output)

refresh_editor()


# Zone Editor — Zones 01, 02 and 03

Output()

Output()

Output()

In [3]:

# ============================================================
# Stage all zones and protocols
# ============================================================

def build_staged_source(protocol):
    config = PROTOCOLS[protocol]

    if read_source(protocol) != original_sources[protocol]:
        raise RuntimeError(
            f"{protocol} source changed externally. "
            "Reload Cell 1 before proceeding."
        )

    staged = original_sources[protocol]

    for zone in ZONE_IDS:
        for name, key_function in zip(
            definition_names(zone, protocol),
            (key_r01, key_r02),
        ):
            replacement = render_definition(
                name,
                rules_by_zone_protocol[zone][protocol],
                key_function,
                protocol,
                zone,
            )

            pattern = (
                r"(?m)^[ \t]*define[ \t]+"
                + re.escape(name)
                + r"[ \t]*=[ \t]*\{[^}]*\}"
            )

            staged, count = re.subn(
                pattern,
                lambda match, replacement=replacement: replacement,
                staged,
                count=1,
                flags=re.DOTALL,
            )

            if count != 1:
                raise RuntimeError(
                    f"Expected one definition for {name}; "
                    f"found {count}."
                )

    return staged


def definition_names(zone, protocol):
    suffix = {
        "TCP": "TCP4",
        "UDP": "UDP4",
        "ICMP": "ICMP4",
    }[protocol]

    return (
        f"Z{zone}_{suffix}_R01",
        f"Z{zone}_{suffix}_R02",
    )


def stage_all_protocols():
    global staged_sources

    errors = []

    for zone in ZONE_IDS:
        for protocol in PROTOCOLS:
            errors.extend(validate_protocol(zone, protocol))

    if errors:
        raise ValueError("\n".join(errors))

    staged_sources = {}

    for protocol, config in PROTOCOLS.items():
        staged = build_staged_source(protocol)
        config["staging"].write_text(staged)
        staged_sources[protocol] = staged

    TEST_ENTRYPOINT.write_text("\n".join([
        "# Generated candidate ruleset for syntax checking.",
        "flush ruleset",
        f'include "{VARIABLES_INCLUDE}"',
        f'include "{CONSTANTS_INCLUDE}"',
        f'include "{PROTOCOLS["ICMP"]["staging"]}"',
        f'include "{PROTOCOLS["UDP"]["staging"]}"',
        f'include "{PROTOCOLS["TCP"]["staging"]}"',
        f'include "{ZONES_INCLUDE}"',
        "",
    ]))


# ============================================================
# Candidate syntax check
# ============================================================

def check_candidate():
    command = [
        "ip", "netns", "exec", NAMESPACE,
        "nft", "-c", "-f", str(TEST_ENTRYPOINT),
    ]

    result = subprocess.run(
        command,
        capture_output=True,
        text=True,
        timeout=30,
        check=False,
    )

    output = "\n".join(
        value.strip()
        for value in (result.stdout, result.stderr)
        if value.strip()
    )

    if result.returncode != 0:
        raise RuntimeError(
            f"Candidate syntax check failed "
            f"(exit {result.returncode}).\n{output}"
        )

    return output or "Candidate passed nft -c."


# ============================================================
# Display proposed changes
# ============================================================

def show_diffs():
    with diff_output:
        clear_output(wait=True)

        for protocol, config in PROTOCOLS.items():
            display(Markdown(f"### Proposed {protocol} changes"))

            diff = difflib.unified_diff(
                original_sources[protocol].splitlines(),
                staged_sources[protocol].splitlines(),
                fromfile=str(config["source"]),
                tofile=str(config["staging"]),
                lineterm="",
            )

            print("\n".join(diff) or "No changes.")


# ============================================================
# Check button
# ============================================================

def run_check(_):
    deploy_button.disabled = True

    with workflow_output:
        clear_output(wait=True)

        try:
            stage_all_protocols()
            output = check_candidate()
            show_diffs()

            print("Zones 01, 02 and 03 were staged.")
            print("TCP, UDP and ICMP candidate check passed.")
            print(f"Test entry point: {TEST_ENTRYPOINT}")
            print(output)

            deploy_button.disabled = False
            set_status("Candidate passed syntax check.")

        except Exception as exc:
            set_status(str(exc), error=True)
            print(f"ERROR: {exc}")


stage_button.on_click(run_check)


# ============================================================
# Deployment and production reload
# ============================================================

def deploy_all(_):
    deploy_button.disabled = True

    with workflow_output:
        clear_output(wait=True)

        try:
            stage_all_protocols()
            output = check_candidate()
            print(output)

            if not PRODUCTION_ENTRYPOINT.is_file():
                raise FileNotFoundError(
                    f"Missing entry point: {PRODUCTION_ENTRYPOINT}"
                )

            for protocol in PROTOCOLS:
                if read_source(protocol) != original_sources[protocol]:
                    raise RuntimeError(
                        f"{protocol} source changed externally. "
                        "Deployment cancelled."
                    )

            for protocol, config in PROTOCOLS.items():
                shutil.copyfile(
                    config["staging"],
                    config["source"],
                )
                print(f"Deployed {protocol}: {config['source']}")

            command = [
                "ip", "netns", "exec", NAMESPACE,
                "nft", "-f", str(PRODUCTION_ENTRYPOINT),
            ]

            result = subprocess.run(
                command,
                capture_output=True,
                text=True,
                timeout=30,
                check=False,
            )

            reload_output = "\n".join(
                value.strip()
                for value in (result.stdout, result.stderr)
                if value.strip()
            )

            if result.returncode != 0:
                raise RuntimeError(
                    "Production reload failed after file replacement. "
                    "The files remain changed.\n" + reload_output
                )

            for protocol in PROTOCOLS:
                original_sources[protocol] = read_source(protocol)

            staged_sources.clear()

            print(
                f"Reloaded {PRODUCTION_ENTRYPOINT} "
                f"in namespace {NAMESPACE}."
            )
            print(reload_output or "Reload succeeded.")

            set_status(
                "Zones 01, 02 and 03 deployed; ruleset reloaded."
            )

        except Exception as exc:
            print(f"ERROR: {exc}")
            set_status(str(exc), error=True)


deploy_button.on_click(deploy_all)
